In [1]:
import pandas as pd
import regex as re
import numpy as np

In [2]:
df = pd.read_csv("Clinical Trials Data, Compiled and Cleaned - [FDA] Drug Trials Snapshots_ All .csv")
df.head()

,BRAND NAME,INDICATION,WOMEN,WHITE,BLACK OR AFRICAN AMERICAN,ASIAN,"ALL OTHER (Aggregated):\nAmerican Indian or Alaska Native (AI/AN), Native Hawaiian or Other Pacific Islander (NH/OPI), Unknown/Unreported",HISPANIC OR LATINO (2017 AND LATER ONLY),UNITED STATES (2017 ONLY),AGE\n65 and OLDER,AGE\n75 and OLDER,AGE\n80 and OLDER,YEAR,NOTES
0,ADDYI,"Treatment of acquired, generalized hypoactive ...",100%,89%,8%,1%,2%,NaN,NaN,0%,0%,0%,2015,NaN
1,ALECENSA,For the treatment of metastatic non-small cell...,55%,74%,2%,18%,7%,NaN,NaN,14%,4%,<1%,2015,NaN
2,ARISTADA,Treatment of schizophrenia,32%,47%,40%,13%,<1%,NaN,NaN,0%,0%,0%,2015,NaN
3,AVYCAZ,Treatment of complicated intra-abdominal infec...,26%,60%,<1%,27%,12%,NaN,NaN,11%,8%,4%,2015,NaN
4,AVYCAZ,Treatment of complicated urinary tract infecti...,74%,60%,5%,10%,25%,NaN,NaN,17%,4%,3%,2015,NaN


In [3]:
# The original column names are very long and contain newlines
# rename to make short, clean names
df.columns = [
    'brand_name', 'indication', 'women', 'white', 'black', 'asian', 'other',
    'hispanic', 'us_only', 'age_65', 'age_75', 'age_80', 'year', 'notes'
]

In [4]:
# create a function to directly parse
# first, see if cell empty -- then return nan
# otherwise, strip and replace all characters with " " + convert to float 
# reused this code from aaliyah's Final Project P1


#: :generated:
#: :model: Claude Sonnet 4.5
#: :prompt: I have a df with columns that store percentage values as string in formats like "45%", "<1%", ">2%": help me crate a simple function that parses the strings and returns them as float values, also handle null values  
#: :changes: Added special cases to first 'if' statement. Add ValueError except clause.
#: :response:
def parse(val):
    #Convert a string like '<1%' to a float.
    if pd.isna(val) or val == "NR" or val == "Not Recorded": #add in cases for NR/not recorded
        return np.nan
    val = str(val).strip().replace('%', '').replace('<', '').replace('>', '')
    try:
        return float(val) # convert cleaned string into float 
    except ValueError:
        return np.nan #return nan instead of crashing for errors 
#: :end:

In [5]:
# Apply function to all percentage columns
pct_cols = ['women', 'white', 'black', 'asian', 'other', 'age_65', 'age_75', 'age_80']
#hispanic, us_only are left out from this as these variables are only collected for one year (2017)
for col in pct_cols:
    df[col] = df[col].apply(parse)

In [6]:
df.dtypes

brand_name     object
indication     object
women         float64
white         float64
black         float64
asian         float64
other         float64
hispanic       object
us_only        object
age_65        float64
age_75        float64
age_80        float64
year            int64
notes          object
dtype: object

In [7]:
# Strip whitespace and normalize to Title Case
df['brand_name'] = df['brand_name'].str.strip().str.title()

In [8]:
#: :generated:
#: :model: Claude Sonnet 4.5
#: :prompt: I have a df with a column that has the description of different clinical trials (pasted in the column of clinical trial descriptions). i need a function that can extract the main disease that the trial is focusing on i.e. breast cancer, hiv etc. 
#: :changes: The name of my column is 'indication'
#: :response:
def extract_disease(text):
    text = str(text).strip()
    low = text.lower()

    # Step 1: strip leading action phrases
    prefixes = [
        r"^complete regimen for the treatment of\s+",
        r"^part of combination treatment\s+",
        r"^for the treatment of\s+",
        r"^for treatment of\s+",
        r"^for the reversal of the effects of\s+",
        r"^for control of\s+",
        r"^for detection of\s+",
        r"^for prevention of\s+",
        r"^for lowering\s+.*?with\s+",
        r"^preventive treatment of\s+",
        r"^prevention of\s+",
        r"^diagnosis of\s+",
        r"^detection of\s+",
        r"^to treat\s+",
        r"^to improve glucose control in adults with\s+",
        r"^to reduce hospitalization from worsening\s+",
        r"^to prevent or reduce bleeding in patients with\s+",
        r"^to increase blood pressure in\s+",
        r"^to prevent\s+",
        r"^treatment for\s+",
        r"^treatment of\s+",
        r"^reversal of the anticoagulant effects of\s+\S+\s+",
        r"^lowering the blood levels of \S+ in adults with\s+",
        r"^decreasing the risk of\s+",
        r"^reduction of risk of\s+",
        r"^parenteral nutrition-associated\s+",
    ]
    for pat in prefixes:
        low = re.sub(pat, '', low, flags=re.IGNORECASE)

    # Step 2: cut at patient-population qualifiers and parentheses
    low = re.split(r'\s+(in adults|in patients|in children|that |when |who |with a |abbreviated|including|a specific type|certain |a type of )\b', low)[0]
    low = re.split(r'\s*\(', low)[0]
    low = re.sub(r'\s+called\s+.*$', '', low)
    low = re.sub(r'\s+that\s+.*$', '', low)
    low = low.strip().rstrip('.,')

    # Step 3: manual overrides for cases regex can't cleanly handle
    overrides = {
        'the effects of':                            'Neuromuscular Blockade Reversal',
        'during emergency situations or':            'Anticoagulation Reversal',
        'certain neuromuscular blocking agents':     'Neuromuscular Blockade Reversal',
        'the signs and symptoms of dry eye disease': 'Dry Eye Disease',
        'the nausea and vomiting':                   'Chemotherapy-Induced Nausea And Vomiting',
        'uric acid levels in the blood of adult patients with gout': 'Gout',
        'a specific form of advanced breast cancer': 'Breast Cancer (ER+/HER2-)',
        'a specific type of severe asthma':          'Eosinophilic Asthma',
        'a specific type of tumors':                 'Neuroendocrine Tumors',
        'a type of bladder cancer':                  'Urothelial Carcinoma',
        'patients with advanced non-small cell lung cancer': 'Non-Small Cell Lung Cancer',
        'patients with hereditary orotic aciduria':  'Hereditary Orotic Aciduria',
        'children with high-risk neuroblastoma':     'Neuroblastoma',
        'adults with pulmonary artery hypertension': 'Pulmonary Arterial Hypertension',
        'adults with low platelet count':            'Immune Thrombocytopenia',
        'adults with low platelet count due to chronic immune thrombocytopenia': 'Immune Thrombocytopenia',
        'adult patients with rheumatoid arthritis':  'Rheumatoid Arthritis',
        'adults with acute myeloid leukemia':        'Acute Myeloid Leukemia',
        'adults with mycosis fungoides or sézary syndrome': 'Mycosis Fungoides / Sézary Syndrome',
        'adults':                                    'Complicated Urinary Tract Infection',
        'women with':                                'Ovarian Cancer',
        'certain patients with high cholesterol':    'Hypercholesterolemia',
        'certain types of advances tissue sarcoma':  'Soft Tissue Sarcoma',
        'symptoms associated with opioid withdrawal during abrupt opioid discontinuation': 'Opioid Withdrawal',
        'seizures in two rare and severe forms of epilepsy': 'Epilepsy',
        'phenylalanine':                             'Phenylketonuria',
        'cholestasis':                               'Parenteral Nutrition-Associated Cholestasis',
        'lysosomal acid lipase':                     'Lysosomal Acid Lipase Deficiency',
        'hypocalcemia along with calcium and vitamin d': 'Hypoparathyroidism',
        'coronary artery blood clot formation':      'PCI-Related Thrombosis Prevention',
        'treatment for double chin':                 'Submental Fat (Double Chin)',
        'delayed phase chemotherapy-induced nausea and vomiting': 'Chemotherapy-Induced Nausea And Vomiting',
        'cytomegalovirus infection in allogeneic hematopoietic stem cell transplant': 'Cytomegalovirus Infection',
        'malaria relapse caused by the parasite, plasmodium vivax': 'Malaria',
        'somatostatin receptor-positive gastroenteropancreatic neuroendocrine tumors': 'Neuroendocrine Tumors',
        'septic or other distributive shock':        'Septic Shock',
        'moderate to severe pain associated with endometriosis': 'Endometriosis',
        'bile acid synthesis disorders due to single enzyme defects': 'Bile Acid Synthesis Disorders',
        'hallucinations and delusions':              "Psychosis (Parkinson's-Related)",
    }

    result = overrides.get(low, low)
    return result.title()
#: :end:

In [9]:
df['disease'] = df['indication'].apply(extract_disease)

In [10]:
df['disease'].nunique()

115

In [11]:
df[['brand_name', 'indication', 'disease']].head(15)

,brand_name,indication,disease
0,Addyi,"Treatment of acquired, generalized hypoactive ...","Acquired, Generalized Hypoactive Sexual Desire..."
1,Alecensa,For the treatment of metastatic non-small cell...,Metastatic Non-Small Cell Lung Cancer
2,Aristada,Treatment of schizophrenia,Schizophrenia
3,Avycaz,Treatment of complicated intra-abdominal infec...,Complicated Intra-Abdominal Infection
4,Avycaz,Treatment of complicated urinary tract infecti...,Complicated Urinary Tract Infection
5,Bridion,For the reversal of the effects of certain neu...,Neuromuscular Blockade Reversal
6,Cholbam,For treatment of bile acid synthesis disorders...,Bile Acid Synthesis Disorders
7,Cholbam,"For treatment of peroxisomal disorders, includ...",Peroxisomal Disorders
8,Corlanor,To reduce hospitalization from worsening heart...,Heart Failure
9,Cosentyx,Treatment of moderate to severe plaque psorias...,Moderate To Severe Plaque Psoriasis


In [12]:
#: :generated:
#: :model: Claude Sonnet 4.5
#: :prompt: I also want to create a function that can then  define a column in the dataset that takes the 'disease' column and groups it into a disease category. I want a cancer, cardiovascular, neurological, infectious
# metabolic, and immune grouping. 
#: :response: Add an option for 'other' diseases
def categorize_disease(indication):
    low = str(indication).lower()
    
    cancer_kws = ['cancer', 'tumor', 'carcinoma', 'leukemia', 'lymphoma',
                  'melanoma', 'myeloma', 'sarcoma', 'neuroblastoma',
                  'mesothelioma', 'glioblastoma', 'neuroendocrine', 'malignant']
    infectious_kws = ['hepatitis', 'hiv', 'infection', 'anthrax', 'chagas',
                      'malaria', 'tuberculosis', 'vaginosis', 'cytomegalovirus',
                      'onchocerciasis', 'impetigo', 'smallpox']
    cardiovascular_kws = ['heart failure', 'cholesterol', 'hypertension',
                          'thromboembolism', 'stroke', 'atrial fibrillation',
                          'thrombosis', 'anticoagul', 'venous', 'coronary',
                          'hyperkalemia', 'shock']
    neurological_kws = ['schizophrenia', 'depression', 'bipolar', 'parkinson',
                        'epilepsy', 'seizure', 'multiple sclerosis', 'huntington',
                        'muscular dystrophy', 'als', 'amyotrophic', 'migraine',
                        'tardive', 'spinal muscular', 'batten', 'neuromuscular']
    metabolic_kws = ['diabetes', 'obesity', 'cholesterol', 'gout', 'osteoporosis',
                     'lysosomal', 'hypophosphatasia', 'mucopolysaccharidosis',
                     'phenylketonuria', 'hypophosphatemia', 'hyperparathyroidism',
                     'orotic aciduria', 'hypocalcemia', 'hypoparathyroidism']
    immune_kws = ['psoriasis', 'rheumatoid arthritis', 'asthma', 'dermatitis',
                  'eczema', 'lupus', 'immunodeficiency', 'thrombocytopenia',
                  'cholangitis', 'peroxisomal', 'mastocytosis']
    
    if any(k in low for k in cancer_kws):
        return 'Cancer'
    if any(k in low for k in infectious_kws):
        return 'Infectious Disease'
    if any(k in low for k in cardiovascular_kws):
        return 'Cardiovascular'
    if any(k in low for k in neurological_kws):
        return 'Neurological'
    if any(k in low for k in metabolic_kws):
        return 'Metabolic / Genetic'
    if any(k in low for k in immune_kws):
        return 'Immune / Inflammatory'
    return 'Other'
#: :end:

In [13]:
df['disease_category'] = df['indication'].apply(categorize_disease)

In [14]:
print(df['disease_category'].value_counts())

Cancer                   39
Other                    26
Infectious Disease       23
Neurological             20
Metabolic / Genetic      18
Immune / Inflammatory    15
Cardiovascular           14
Name: disease_category, dtype: int64


In [15]:
df = df[[
    'brand_name', 'indication', 'disease', 'disease_category', 'year',
    'women', 'white', 'black', 'asian', 'other',
    'hispanic', 'us_only',
    'age_65', 'age_75', 'age_80'
]]

df.head()

,brand_name,indication,disease,disease_category,year,women,white,black,asian,other,hispanic,us_only,age_65,age_75,age_80
0,Addyi,"Treatment of acquired, generalized hypoactive ...","Acquired, Generalized Hypoactive Sexual Desire...",Other,2015,100.0,89.0,8.0,1.0,2.0,NaN,NaN,0.0,0.0,0.0
1,Alecensa,For the treatment of metastatic non-small cell...,Metastatic Non-Small Cell Lung Cancer,Cancer,2015,55.0,74.0,2.0,18.0,7.0,NaN,NaN,14.0,4.0,1.0
2,Aristada,Treatment of schizophrenia,Schizophrenia,Neurological,2015,32.0,47.0,40.0,13.0,1.0,NaN,NaN,0.0,0.0,0.0
3,Avycaz,Treatment of complicated intra-abdominal infec...,Complicated Intra-Abdominal Infection,Infectious Disease,2015,26.0,60.0,1.0,27.0,12.0,NaN,NaN,11.0,8.0,4.0
4,Avycaz,Treatment of complicated urinary tract infecti...,Complicated Urinary Tract Infection,Infectious Disease,2015,74.0,60.0,5.0,10.0,25.0,NaN,NaN,17.0,4.0,3.0


In [16]:
print(df.isnull().sum())

brand_name            0
indication            0
disease               0
disease_category      0
year                  0
women                 0
white                 2
black                 3
asian                 3
other                 0
hispanic             80
us_only             107
age_65                8
age_75              103
age_80              103
dtype: int64


In [17]:
output_path = 'clinical_trials_cleaned.csv'
df.to_csv(output_path, index=False)

In [18]:
print(f"Saved: {output_path}")
print(f"Shape: {df.shape}")

Saved: clinical_trials_cleaned.csv
Shape: (155, 15)


In [19]:
# Verify saved correctly
verify = pd.read_csv(output_path)

In [20]:
verify.head()

,brand_name,indication,disease,disease_category,year,women,white,black,asian,other,hispanic,us_only,age_65,age_75,age_80
0,Addyi,"Treatment of acquired, generalized hypoactive ...","Acquired, Generalized Hypoactive Sexual Desire...",Other,2015,100.0,89.0,8.0,1.0,2.0,NaN,NaN,0.0,0.0,0.0
1,Alecensa,For the treatment of metastatic non-small cell...,Metastatic Non-Small Cell Lung Cancer,Cancer,2015,55.0,74.0,2.0,18.0,7.0,NaN,NaN,14.0,4.0,1.0
2,Aristada,Treatment of schizophrenia,Schizophrenia,Neurological,2015,32.0,47.0,40.0,13.0,1.0,NaN,NaN,0.0,0.0,0.0
3,Avycaz,Treatment of complicated intra-abdominal infec...,Complicated Intra-Abdominal Infection,Infectious Disease,2015,26.0,60.0,1.0,27.0,12.0,NaN,NaN,11.0,8.0,4.0
4,Avycaz,Treatment of complicated urinary tract infecti...,Complicated Urinary Tract Infection,Infectious Disease,2015,74.0,60.0,5.0,10.0,25.0,NaN,NaN,17.0,4.0,3.0


In [21]:
verify.shape


(155, 15)